# Telco Churn Prediction with Spark MLlib - Fixed Version

This notebook addresses the main issues in the original submission: stratified splitting, SMOTE-based class balancing, consistent feature representation for both models, F1/precision/recall reporting, SHAP interpretability, and a cleaner Spark shutdown.

In [ ]:
import sys
import subprocess

def ensure_packages(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *packages])

try:
    import imblearn  # noqa: F401
    import shap  # noqa: F401
except Exception:
    ensure_packages(['imbalanced-learn', 'scikit-learn', 'shap', 'matplotlib'])

In [ ]:
import os
import numpy as np

os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml.linalg import Vectors

java_home = r'C:\Program Files\Java\jdk-21.0.11'
if os.path.exists(java_home):
    os.environ['JAVA_HOME'] = java_home
    os.environ['PATH'] = java_home + r'\bin;' + os.environ.get('PATH', '')

spark = (
    SparkSession.builder
    .appName('TelcoChurnMLlibFixed')
    .config('spark.driver.memory', '2g')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .config('spark.python.worker.reuse', 'true')
    .config('spark.python.use.daemon', 'true')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark session ready')

## Load and clean the data

In [ ]:
data_path = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
raw_df = spark.read.option('header', True).csv(data_path)
print('Rows:', raw_df.count())
print('Columns:', len(raw_df.columns))
raw_df.printSchema()

In [ ]:
df = (
    raw_df
    .withColumnRenamed('customerID', 'customer_id')
    .withColumn('label', F.when(F.col('Churn') == 'Yes', F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn('SeniorCitizen', F.col('SeniorCitizen').cast(IntegerType()))
    .withColumn('tenure', F.col('tenure').cast(DoubleType()))
    .withColumn('MonthlyCharges', F.col('MonthlyCharges').cast(DoubleType()))
    .withColumn('TotalCharges', F.when(F.trim(F.col('TotalCharges')) == '', F.lit(0.0)).otherwise(F.col('TotalCharges').cast(DoubleType())))
    .drop('Churn')
)
df = df.withColumn('charge_per_tenure', F.when(F.col('tenure') > 0, F.col('TotalCharges') / F.col('tenure')).otherwise(F.col('MonthlyCharges')))
df = df.withColumn('tenure_group', F.when(F.col('tenure') <= 12, 'Short').when(F.col('tenure') <= 48, 'Medium').otherwise('Long'))
df.groupBy('label').count().orderBy('label').show()

## Stratified train/test split

`sampleBy` is used so the train set keeps the label ratio approximately intact, and a left-anti join removes overlap with the test set.

In [ ]:
fractions = {0.0: 0.8, 1.0: 0.8}
train_df = df.stat.sampleBy('label', fractions, seed=42)
train_ids = train_df.select('customer_id').distinct()
test_df = df.join(train_ids, on='customer_id', how='left_anti')
print('Train label counts')
train_df.groupBy('label').count().orderBy('label').show()
print('Test label counts')
test_df.groupBy('label').count().orderBy('label').show()

## Feature preparation

Both models use the same scaled `features` column so the comparison is fair.

In [ ]:
categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group'
]
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'charge_per_tenure']

indexers = [StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep') for c in categorical_cols]
encoder = OneHotEncoder(
    inputCols=[f'{c}_idx' for c in categorical_cols],
    outputCols=[f'{c}_ohe' for c in categorical_cols],
    dropLast=False,
    handleInvalid='keep'
)
assembler = VectorAssembler(
    inputCols=numeric_cols + [f'{c}_ohe' for c in categorical_cols],
    outputCol='features_raw',
    handleInvalid='keep'
)
preprocess = Pipeline(stages=indexers + [encoder, assembler])
preprocess_model = preprocess.fit(train_df)
train_prepared = preprocess_model.transform(train_df)
test_prepared = preprocess_model.transform(test_df)

scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=False, withStd=True)
scaler_model = scaler.fit(train_prepared)
train_scaled = scaler_model.transform(train_prepared)
test_scaled = scaler_model.transform(test_prepared)

print('Feature vector prepared')

## SMOTE on the training set only

The minority class is oversampled after preprocessing, using only the training split, so the test set stays untouched.

In [ ]:
train_pd = train_scaled.select(vector_to_array('features').alias('features_arr'), 'label').toPandas()
X = np.vstack(train_pd['features_arr'].values)
y = train_pd['label'].astype(int).values
print('Before SMOTE:', np.bincount(y))

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print('After SMOTE:', np.bincount(y_res))

train_resampled = spark.createDataFrame(
    [(Vectors.dense(row.tolist()), float(label)) for row, label in zip(X_res, y_res)],
    ['features', 'label']
)
train_resampled = train_resampled.cache()
print('Resampled training frame prepared')

## Train the models

Logistic Regression and Random Forest are both trained on the same balanced feature matrix.

In [ ]:
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=50)
lr_model = lr.fit(train_resampled)

rf = RandomForestClassifier(featuresCol='features', labelCol='label', seed=42, probabilityCol='probability')
rf_model = rf.fit(train_resampled)
print('Models trained')

## Evaluate on the untouched test split

F1 is reported because accuracy alone is not reliable on imbalanced churn data.

In [ ]:
auc_eval = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
acc_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
prec_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision')
recall_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall')

lr_preds = lr_model.transform(test_scaled)
rf_preds = rf_model.transform(test_scaled)

results = [
    ('LogisticRegression', auc_eval.evaluate(lr_preds), acc_eval.evaluate(lr_preds), f1_eval.evaluate(lr_preds), prec_eval.evaluate(lr_preds), recall_eval.evaluate(lr_preds)),
    ('RandomForest', auc_eval.evaluate(rf_preds), acc_eval.evaluate(rf_preds), f1_eval.evaluate(rf_preds), prec_eval.evaluate(rf_preds), recall_eval.evaluate(rf_preds))
]
results_df = spark.createDataFrame(results, ['model', 'auc_roc', 'accuracy', 'f1', 'precision', 'recall'])
results_df.show(truncate=False)

print('Logistic Regression confusion summary')
lr_preds.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()
print('Random Forest confusion summary')
rf_preds.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

## Interpretability

Spark RF feature importances are shown first, then SHAP is computed on a sklearn surrogate trained on the same balanced matrix.

In [ ]:
feature_attrs = train_prepared.schema['features_raw'].metadata['ml_attr']['attrs']
all_attrs = [attr for group in feature_attrs.values() for attr in group]
feature_names = [attr.get('name', f'feature_{attr["idx"]}') for attr in sorted(all_attrs, key=lambda attr: attr['idx'])]
importance_rows = sorted(
    [(name, float(score)) for name, score in zip(feature_names, rf_model.featureImportances.toArray())],
    key=lambda item: item[1],
    reverse=True
)
spark.createDataFrame(importance_rows[:15], ['feature', 'importance']).show(truncate=False)

from sklearn.ensemble import RandomForestClassifier as SkRandomForestClassifier
import shap

rng = np.random.RandomState(42)
background_size = min(200, X_res.shape[0])
background = X_res[rng.choice(X_res.shape[0], size=background_size, replace=False)]
sk_rf = SkRandomForestClassifier(n_estimators=100, random_state=42)
sk_rf.fit(X_res, y_res)
explainer = shap.TreeExplainer(sk_rf)
shap_values = explainer.shap_values(background)

if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], background, feature_names=feature_names, show=False)
else:
    shap.summary_plot(shap_values, background, feature_names=feature_names, show=False)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## Top churn-risk customers

In [ ]:
best_model_name = results_df.orderBy(F.desc('auc_roc')).first()['model']
best_preds = rf_preds if best_model_name == 'RandomForest' else lr_preds
best_preds.select('customer_id', F.round(vector_to_array('probability')[1], 4).alias('churn_probability'), 'prediction').orderBy(F.desc('churn_probability')).show(10, truncate=False)
print('Best model by AUC:', best_model_name)

In [ ]:
try:
    spark.stop()
    print('Spark stopped successfully')
except ConnectionResetError:
    print('Ignored ConnectionResetError during Spark shutdown')
except Exception as e:
    print('Spark stop raised:', type(e).__name__, e)

## Notes

- The training split is stratified with `sampleBy` and then isolated from the test split using a left-anti join.
- SMOTE is applied only to the training data.
- Both classifiers use the same scaled `features` column.
- The evaluation section reports AUC, accuracy, F1, precision, and recall.
- SHAP is generated through a sklearn surrogate so the explanation is practical inside a notebook.
- Spark shutdown is wrapped so a socket cleanup error does not clutter the output.